In [0]:
from google.colab import files
import io
import os

import csv

import statsmodels.api as sm
import math
from sklearn.linear_model import LinearRegression

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import linalg as LA
import operator
from xlwt import Workbook
import xlwt

In [2]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)
root_path = 'drive/My Drive/ECI Research/Spring 2020/Data/'

Go to this URL in a browser: https://accounts.google.com/o/oauth2/auth?client_id=947318989803-6bn6qk8qdgf4n4g3pfee6491hc0brc4i.apps.googleusercontent.com&redirect_uri=urn%3aietf%3awg%3aoauth%3a2.0%3aoob&response_type=code&scope=email%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdocs.test%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive.photos.readonly%20https%3a%2f%2fwww.googleapis.com%2fauth%2fpeopleapi.readonly

Enter your authorization code:
··········
Mounted at /content/drive/


# I. Import Data

In [4]:
#root_path = 'drive/My Drive/ECI Research/Spring 2020/Data'

# Read input
uploaded = files.upload()
df = pd.read_csv(io.BytesIO(uploaded['2015clean.csv']))

uploaded = files.upload()
df2 = pd.read_csv(io.BytesIO(uploaded['2018test.csv']))

Saving 2015clean.csv to 2015clean.csv


KeyError: ignored

In [5]:
raw_matrix = df.to_numpy()
raw_matrix[0][1::]

array([479039774, 1533140749, 2027579611, 3380143043.0, 374334464,
       1708529011.0, 1405300551.0, 506813340, 1853225282.0, 272988749,
       3931071926.0, 2104089906.0, 3278996259.0, 7817434, 1158283825.0,
       706887964, 6786931358.0, 1000144551.0], dtype=object)

In [6]:
raw_matrix = df.to_numpy()
matrix = [list(df.columns)]

for row in raw_matrix:
  if np.min(row[1::]) > 0:
    matrix.append(list(row))
matrix

[['Name',
  'Animals and By-products',
  'Vegetables, Fruits and Plants',
  'Food and Preparations',
  'Energy Drilling and Mining',
  'Chemicals',
  'Chemical Manufacturing',
  'Raw Manufacturing',
  'Wood Products',
  'Clothing and Accessories',
  'Textile',
  'Metals',
  'Machinery and Construction',
  'Transportation',
  'Weapons',
  'Daily Instruments',
  'Precious Stones',
  'High-complexity Products',
  'Services and Utilities '],
 ['Bulgaria',
  479039774,
  1533140749,
  2027579611,
  3380143043.0,
  374334464,
  1708529011.0,
  1405300551.0,
  506813340,
  1853225282.0,
  272988749,
  3931071926.0,
  2104089906.0,
  3278996259.0,
  7817434,
  1158283825.0,
  706887964,
  6786931358.0,
  1000144551.0],
 ['Myanmar',
  456418904,
  1805285809,
  1198050479,
  4861069945.0,
  1002864,
  1448902.0,
  329566051.0,
  84799631,
  937645994.0,
  22121673,
  405436330.0,
  22371650.0,
  17015745.0,
  13599,
  44712432.0,
  629999072,
  40965740.0,
  613660288.0],
 ['Burundi',
  220635,

In [7]:
matrix[0][1::]

['Animals and By-products',
 'Vegetables, Fruits and Plants',
 'Food and Preparations',
 'Energy Drilling and Mining',
 'Chemicals',
 'Chemical Manufacturing',
 'Raw Manufacturing',
 'Wood Products',
 'Clothing and Accessories',
 'Textile',
 'Metals',
 'Machinery and Construction',
 'Transportation',
 'Weapons',
 'Daily Instruments',
 'Precious Stones',
 'High-complexity Products',
 'Services and Utilities ']

# ECI & PCI

MCP

In [0]:
#compute mcp
product_list = matrix[0][1::]  # return product names
country_list = []   
for i in range(1,len(matrix)):
  country_list.append(matrix[i][0])  # search for country names
  
country_sum = []
for i in range(1,len(matrix)):
  country_sum.append(np.sum(matrix[i][1::])) 
  
product_sum = []
for i in range(1,len(matrix[0])):
  product = 0
  for j in range(1,len(matrix)):
      product += matrix[j][i]
  product_sum.append(product)
total_sum = np.sum(product_sum)
  
mcp = []
  
for i in range (1,len(matrix)):
  country_row = []
  for j in range(1,len(matrix[i])):
    try:
      percentage = matrix[i][j]/country_sum[i-1]
    except:
      print(country_list[i-1])
    total_percentage = product_sum[j-1]/total_sum
    if percentage > total_percentage:
      country_row.append(1)
    else:
      country_row.append(0)
  mcp.append(country_row)


In [9]:
product_sum

[353097712764,
 440592576863,
 536953103819,
 1720825963415.0,
 439810548931,
 973015398276.0,
 914487741818.0,
 306298850117,
 648953390140.0,
 180156185793,
 976677289190.0,
 1945861142046.0,
 3737779298242.0,
 10051639450,
 1091261590800.0,
 786315290685,
 6903284712764.0,
 610471583354.0]

MCC

In [0]:
#compute mcc

mcp2 = mcp
country_sum = []
for i in range(0,len(mcp)):
  country_sum.append(np.sum(mcp[i]))
  
product_sum = []
for i in range(0,len(mcp[0])):
  product = 0
  for j in range(0,len(mcp)):
    product += mcp[j][i]
  product_sum.append(product)
total_sum = np.sum(product_sum)
  
mcc = []
for i in range(0,len(mcp)):
  country_row = []
  for j in range(0,len(mcp)):
    temp_sum = 0
    for k in range(0,len(mcp[0])):
      try:
        value = (mcp[i][k] * mcp2[j][k]/(country_sum[i]*product_sum[k]))
        if np.isinf(value) or np.isnan(value):
          temp_sum += 0
        else:
          temp_sum += value
      except:
        pass
    country_row.append(temp_sum)
  mcc.append(country_row)


In [11]:
mcc[0]

[0.01987213506513804,
 0.009505490602037916,
 0.006030430946737593,
 0.015163383671604037,
 0.004809634117367546,
 0.008641437723839084,
 0.012212866295267655,
 0.009885885584394212,
 0.008641437723839084,
 0.00714451242120875,
 0.0076177325340391795,
 0.0075688924851991315,
 0.011675960043405209,
 0.003304130162703379,
 0.005234698097601323,
 0.003252247488101534,
 0.010508722263264202,
 0.008012881556237382,
 0.003850728697276012,
 0.014713404906407882,
 0.0027637721755368813,
 0.011036395707032362,
 0.0015873015873015873,
 0.009111867831365965,
 0.008362137597991643,
 0.004853960358502298,
 0.0022634271099744247,
 0.009674624830883882,
 0.00571170297204339,
 0.00913746946987083,
 0.0056990881458966565,
 0.004783193080878915,
 0.014217373160376136,
 0.0027637721755368813,
 0.00439108668444251,
 0.014687803267903017,
 0.00430223371399842,
 0.003801888648435963,
 0.0,
 0.0027263007840342126,
 0.003876330335780876,
 0.010573522032972573,
 0.011473187171695801,
 0.003214616096207216,
 0.

MPP

In [0]:
# compute mpp

mcp2 = mcp
country_sum = []
for i in range(0,len(mcp)):
  country_sum.append(np.sum(mcp[i]))
  
product_sum = []
for i in range(0,len(mcp[0])):
  product = 0
  for j in range(0,len(mcp)):
    product += mcp[j][i]
  product_sum.append(product)
total_sum = np.sum(product_sum)
  
mpp = []
for i in range(0,len(mcp[0])):
  product_row = []
  for j in range(0,len(mcp[0])):
    temp_sum = 0
    for k in range(0,len(mcp)):
      try:
        temp_sum += (mcp[k][i] * mcp2[k][j]/(country_sum[k]*product_sum[i]))
      except:
        pass
    product_row.append(temp_sum)
  mpp.append(product_row)


In [13]:
n = np.array(mpp)
n.shape

(18, 18)

ECI

In [0]:
#compute ECI
mcc = np.array(mcc)
  
e_vals, e_vecs = LA.eig(mcc)
eigenvectors = e_vecs.tolist()
eigen = []
for eigenvector in eigenvectors:
  temp = []
  for col in eigenvector:
    temp.append(col.real)
  eigen.append(temp)
  
vector = []
for i in range(0,len(eigen)):
  vector.append(eigen[i][1])
  
eci = []
if vector[0] < 0:
  for item in vector:
    eci.append((item-np.mean(vector))/np.std(vector))
else:
  for item in vector:
    eci.append((-item+np.mean(vector))/np.std(vector))


PCI

In [0]:
#compute PCI
mpp = np.array(mpp)
e_vals, e_vecs = LA.eig(mpp)
eigenvectors = e_vecs.tolist()
eigen = []
for eigenvector in eigenvectors:
  temp = []
  for col in eigenvector:
    temp.append(col.real)
  eigen.append(temp)
  
vector = []
for i in range(0,len(eigen)):
  vector.append(eigen[i][1])
  
pci = []
if vector[len(vector)-2] > 0:
  for item in vector:
    pci.append((item-np.mean(vector))/np.std(vector))
else:
  for item in vector:
    pci.append((-item+np.mean(vector))/np.std(vector))


In [27]:

#check ECI
eci_dict = {}
for i in range(len(country_list)):
  eci_dict[country_list[i]]=eci[i]
sorted_eci_dict = sorted(eci_dict.items(), key=operator.itemgetter(1))

#print(sorted_eci_dict)
for item in sorted_eci_dict:
  print(item)

('Angola', -1.363857097347638)
('Azerbaijan', -1.2212320488458155)
('Mongolia', -1.1809048042868842)
('Ethiopia', -1.1179741975429245)
('Burkina Faso', -1.0858214386190013)
('Ecuador', -1.0782709238731965)
('Rwanda', -1.0782709238731965)
('Paraguay', -1.0164552347089655)
('French Polynesia', -1.0068743805045857)
('Nicaragua', -0.9235892155767497)
('United Republic of Tanzania', -0.9089897010433854)
('Zimbabwe', -0.8777988727412862)
('Lesotho', -0.8703010543963785)
('Plurinational State of Bolivia', -0.8567579293636401)
('Namibia', -0.8492642192115948)
('Armenia', -0.845053807466767)
("C̫te d'Ivoire", -0.802593386493637)
('Myanmar', -0.8017495374732023)
('Honduras', -0.780785779898434)
('United Arab Emirates', -0.7732781216529272)
('Mauritius', -0.7518187275795781)
('Seychelles', -0.7392529051833783)
('Belize', -0.7392529051833783)
('Kyrgyzstan', -0.73163582562343)
('Argentina', -0.7277357777898023)
('South Africa', -0.7243838226910114)
('Botswana', -0.7235218841677468)
('Bangladesh', -

# Proximity, Distance, and ADI

Proximity

In [0]:

proximity = []
for i in range(0,len(mcp[0])):
  product_row = []
  for j in range(0,len(mcp[0])):
    temp_sum = 0
    for k in range(0,len(mcp)):
      try:
        temp_sum += mcp[k][i] * mcp[k][j]
      except:
        pass
    p0 = product_sum[i]
    p1 = product_sum[j]
    if p0 < p1:
      large = p1
    else:
      large = p0
    product_row.append(temp_sum/large)
  proximity.append(product_row)


In [38]:
proximity

[[1.0,
  0.6352941176470588,
  0.6304347826086957,
  0.4857142857142857,
  0.18571428571428572,
  0.38571428571428573,
  0.4857142857142857,
  0.5285714285714286,
  0.37142857142857144,
  0.2714285714285714,
  0.44285714285714284,
  0.08571428571428572,
  0.14285714285714285,
  0.22857142857142856,
  0.2,
  0.3142857142857143,
  0.08571428571428572,
  0.21428571428571427],
 [0.6352941176470588,
  1.0,
  0.7717391304347826,
  0.5058823529411764,
  0.15294117647058825,
  0.35294117647058826,
  0.4823529411764706,
  0.5882352941176471,
  0.4117647058823529,
  0.2235294117647059,
  0.4470588235294118,
  0.03529411764705882,
  0.12941176470588237,
  0.25882352941176473,
  0.18823529411764706,
  0.38823529411764707,
  0.08235294117647059,
  0.17647058823529413],
 [0.6304347826086957,
  0.7717391304347826,
  1.0,
  0.44565217391304346,
  0.16304347826086957,
  0.41304347826086957,
  0.5217391304347826,
  0.5760869565217391,
  0.40217391304347827,
  0.2391304347826087,
  0.4891304347826087,
  

Distance

In [0]:
#compute distance

distance = []
for i in range(0,len(mcp)):
  product_row = []
  for j in range(0,len(mcp[0])):
    sum1 = 0
    for k in range(0,len(mcp[0])):
      if k != j:
        try:
          sum1 += mcp[i][k]*proximity[j][k]
        except:
          pass
    sum2 = 0
    for k in range(0,len(mcp[0])):
      if k != j:
        try:
          sum2 += proximity[j][k]
        except:
          pass
    product_row.append(1-sum1/sum2)
  distance.append(product_row)


In [37]:
np.array(distance).shape

(138, 18)

ADI

In [0]:
def compute_adi(df2,country_list,distance):
  raw_matrix = df2.to_numpy()
  matrix = [list(df2.columns)]
  for row in raw_matrix:
    if sum(row[1::]) > 0:
      matrix.append(list(row))

  country_list2,product_list2,mcp2,product_sum2,country_sum2 = compute_mcp(matrix)
  proximity2 = compute_proximity(mcp2,product_sum2)
  distance2 = compute_distance(mcp2,proximity2)

  adi = {}
  for i in range(len(country_list)):
    for j in range(len(country_list2)):
      if country_list[i] == country_list2[j]:
        adi[country_list[i]] = np.array(distance2[j]) - np.array(distance[i])
  return adi, country_list2,product_list2 

In [0]:
proximity = compute_proximity(mcp,product_sum)
distance = compute_distance(mcp,proximity)
adi, country_list2,product_list2  = compute_adi(df2,country_list,distance)

In [0]:
print(adi)

{'Bulgaria': array([ 0.01331032, -0.13237679, -0.10490817, -0.0662309 , -0.09041049,
       -0.04061489, -0.05051162, -0.12496825, -0.09713446, -0.14711222,
       -0.07373076,  0.01709885,  0.01553725, -0.09803242, -0.04361058,
       -0.06323063,  0.01132976, -0.02769632]), 'Myanmar': array([-0.04654753, -0.03656641, -0.05107872, -0.07025321, -0.00555016,
       -0.02982419, -0.07098692, -0.03763471, -0.06744899, -0.05567328,
        0.01668589, -0.02947847,  0.02876462, -0.05261763, -0.05603994,
       -0.03764442, -0.00861929, -0.00907569]), 'Burundi': array([0.04181738, 0.06015388, 0.06691088, 0.1001728 , 0.06447429,
       0.0784399 , 0.02973522, 0.0783816 , 0.07779229, 0.05954935,
       0.08434001, 0.1585182 , 0.18247876, 0.02402839, 0.14286332,
       0.0464686 , 0.18138451, 0.02697743]), 'Belarus': array([-0.12874665, -0.1370263 ,  0.01234414, -0.11699014, -0.07642184,
       -0.12429455, -0.12501997, -0.10710626, -0.14322129, -0.10097551,
       -0.12439294, -0.08112648, -0.

IV. Sequential Strategy

Opportunity Gain

In [0]:
def compute_opportunity_gain(mcp,proximity,pci):
  opportunity_gain = []
  for i in range(0,len(mcp)):
    country_row = []
    for j in range(0,len(mcp[0])):
      gain = 0
      for k in range(0,len(mcp[0])):
        temp = 0
        for l in range(0,len(mcp[0])):
          temp += proximity[l][k]
        gain += (proximity[j][k]/temp)*(1-mcp[i][k])*pci[k]
      gain = gain - (1-distance[i][j]) * pci[j]
      if mcp[i][j] == 1:
        country_row.append(0)
      else:
        country_row.append(gain)
    opportunity_gain.append(country_row)
    
  opportunity_gain = np.array(opportunity_gain)
  return opportunity_gain

Marginal Opportunity Gain

In [0]:
def compute_marginal_opportunity_gain(mcp,opportunity_gain,country_sum,product_sum,matrix):
  marginal = []
  total_sum = np.sum(product_sum)
  for i in range(0,len(mcp)):
    country_row = []
    for j in range(0,len(mcp[0])):
      if mcp[i][j] == 1:
        country_row.append(0)
      else:
        export = matrix[i+1][j+1]
        country_export = country_sum[i]
        product_export = product_sum[j]
        export_increase = (product_export*country_export/total_sum)-export
        country_row.append(opportunity_gain[i][j]*country_export/export_increase)
    marginal.append(country_row)
    
  marginal = np.array(marginal)
  return marginal

In [0]:
opportunity_gain = compute_opportunity_gain(mcp,proximity,pci)
marginal_opportunity_gain = compute_marginal_opportunity_gain(mcp,opportunity_gain,country_sum,product_sum,matrix)

In [0]:
# For computation of sequential strategies, refer to sequential strategy.py

In [0]:
# For sensitivity analysis, refer to sensitivity.ipynb

In [0]:
# For computation of SOI, soi.ipynb

# SDV & SIDS

In [0]:
def read_country(filename):
    f = open(filename,'r', encoding='utf-8')
    reader = csv.reader(f)
    raw_matrix = []
    for row in reader:
        temp = []
        for i in range(0,len(row)):
            try:
                temp.append(float(row[i]))
            except ValueError:
                temp.append(row[i])
        raw_matrix.append(temp)  
    f.close()
    return raw_matrix

def rank(score_dict):
    rank_list = []
    score_list = []
    for item in score_dict:
        score_list.append(score_dict[item])
    while score_list:
        maximum = max(score_list)
        for item in score_dict:
            if score_dict[item] == maximum:
                rank_list.append(item)
        score_list = [score for score in score_list if score!= maximum]
    return rank_list


def sdv_and_sids(country, year):
    #Read the raw matrix
    raw_matrix = read_country(f"{year}.csv")
    raw_matrix = [row for row in raw_matrix if np.all(row[1:]==np.zeros(len(row)-1)) == False]
    #compute mcp
    product_list = raw_matrix[0][1::]
    country_list = []
    for i in range(1,len(raw_matrix)):
        country_list.append(raw_matrix[i][0])
    
    country_sum = []
    for i in range(1,len(raw_matrix)):
        country_sum.append(np.sum(raw_matrix[i][1::]))
    
    product_sum = []
    for i in range(1,len(raw_matrix[0])):
        product = 0
        for j in range(1,len(raw_matrix)):
            product += raw_matrix[j][i]
        product_sum.append(product)
    total_sum = np.sum(product_sum)
    
    #print(product_list)
    #print(country_list)

    mcp = []
    
    for i in range (1,len(raw_matrix)):
        country_row = []
        for j in range(1,len(raw_matrix[i])):
            percentage = raw_matrix[i][j]/country_sum[i-1]
            total_percentage = product_sum[j-1]/total_sum
            if percentage > total_percentage:
                country_row.append(1)
            else:
                country_row.append(0)
        mcp.append(country_row)
    
    #compute mpp
    country_sum2 = []
    for i in range(0,len(mcp)):
        country_sum2.append(np.sum(mcp[i]))
    
    product_sum2 = []
    for i in range(0,len(mcp[0])):
        product = 0
        for j in range(0,len(mcp)):
            product += mcp[j][i]
        product_sum2.append(product)
    
    mpp = []
    for i in range(0,len(mcp[0])):
        product_row = []
        for j in range(0,len(mcp[0])):
            temp_sum = 0
            for k in range(0,len(mcp)):
                try:
                    temp_sum += (mcp[k][i] * mcp[k][j]/(country_sum2[k]*product_sum2[i]))
                except:
                    pass
            product_row.append(temp_sum)
        mpp.append(product_row)
    
    #compute PCI
    mpp = np.array(mpp)
    
    e_vals, e_vecs = LA.eig(mpp)
    eigenvectors = e_vecs.tolist()
    eigen = []
    for eigenvector in eigenvectors:
        temp = []
        for col in eigenvector:
            temp.append(col.real)
        eigen.append(temp)
    
    vector = []
    for i in range(0,len(eigen)):
        vector.append(eigen[i][1])
    
    pci = []
    if vector[len(vector)-2] > 0:
        for item in vector:
            pci.append((item-np.mean(vector))/np.std(vector))
    else:
        for item in vector:
            pci.append((-item+np.mean(vector))/np.std(vector))
    
    #compute proximity
    proximity = []
    for i in range(0,len(mcp[0])):
        product_row = []
        for j in range(0,len(mcp[0])):
            temp_sum = 0
            for k in range(0,len(mcp)):
                try:
                    temp_sum += mcp[k][i] * mcp[k][j]
                except:
                    pass
            p0 = product_sum2[i]
            p1 = product_sum2[j]
            if p0 < p1:
                large = p1
            else:
                large = p0
            product_row.append(temp_sum/large)
        proximity.append(product_row)
    
    #compute actual distance
    distance = []
    for i in range(0,len(mcp)):
        product_row = []
        for j in range(0,len(mcp[0])):
            sum1 = 0
            for k in range(0,len(mcp[0])):
                try:
                    sum1 += (1-mcp[i][k])*proximity[j][k]
                except:
                    pass
            sum2 = 0
            for k in range(0,len(mcp[0])):
                try:
                    sum2 += proximity[j][k]
                except:
                    pass
            if mcp[i][j] == 1:
                product_row.append(0)
            else:
                product_row.append(sum1/sum2)
        distance.append(product_row)
    
    #compute opportunity gain
    opportunity_gain = []
    for i in range(0,len(mcp)):
        country_row = []
        for j in range(0,len(mcp[0])):
            gain = 0
            for k in range(0,len(mcp[0])):
                temp = 0
                for l in range(0,len(mcp[0])):
                    temp += proximity[l][k]
                gain += (proximity[j][k]/temp)*(1-mcp[i][k])*pci[k]
            gain = gain - (1-distance[i][j]) * pci[j]
            if mcp[i][j] == 1:
                country_row.append(0)
            else:
                country_row.append(gain)
        opportunity_gain.append(country_row)
        
    #compute marginal opportunity gain
    marginal = []
    for i in range(0,len(mcp)):
        country_row = []
        for j in range(0,len(mcp[0])):
            if mcp[i][j] == 1:
                country_row.append(0)
            else:
                export = raw_matrix[i+1][j+1]
                country_export = country_sum[i]
                product_export = product_sum[j]
                export_increase = (product_export*country_export/total_sum)-export
                country_row.append(opportunity_gain[i][j]*country_export/export_increase)
        marginal.append(country_row)
    
    
    
    
    
    #computing sequential strategy
    strategy = []
    
    for i in range(0,len(country_list)):
        if country in country_list[i]:
            country_index = i
    og = np.copy(opportunity_gain[country_index])
    sdv = dict(zip(product_list, og))
    sdv = sorted(sdv.items(), key=operator.itemgetter(1), reverse=True)
    sdv_colors = []
    for i in range(len(sdv)):
        if sdv[i][1]>0:
          sdv_colors.append(44)
        elif sdv[i][1]==0:
          sdv_colors.append(27)
        else:
          sdv_colors.append(1)
    #print("Before Iteration: ")
    #for i in range(0,len(marginal[country_index])):
    #    print(product_list[i], ": ", marginal[country_index][i])
    #print(" ")
    
    temp = []
    temp.append("Before Iteration")
    posi = {}
    zero = []
    nega = {}
    colors = []
    color = [1]
    for i in range(0,len(marginal[country_index])):
        if marginal[country_index][i] > 0:
            posi[product_list[i]] = marginal[country_index][i]
        elif marginal[country_index][i] == 0:
            zero.append(product_list[i])
        else:
            nega[product_list[i]] = marginal[country_index][i]
    posi = rank(posi)
    nega = rank(nega)
    temp.extend(posi)
    color.extend([44 for i in range(len(posi))])
    temp.extend(zero)
    color.extend([27 for i in range(len(zero))])
    temp.extend(nega)
    color.extend([1 for i in range(len(nega))])
    strategy.append(temp)
    colors.append(color)
    
    ite_round = 0
    prev_industry = []
    
    while max(marginal[country_index]) > 0 and ite_round < 20:
        ite_round += 1
        
        value = max(marginal[country_index])
        for i in range(0,len(marginal[0])):
            if marginal[country_index][i] == value:
                industry_index = i
        industry = product_list[industry_index]
        prev_industry.append(industry_index)
        
        #calculate new export
        export = raw_matrix[country_index+1][industry_index+1]
        country_export = country_sum[country_index]
        product_export = product_sum[industry_index]
        required_export = product_export*country_export/total_sum
        export_increase = required_export-export
    
        #print("increase ", industry, " in ", country, " by ", export_increase, " USD")
    
        raw_matrix[country_index+1][industry_index+1] = required_export*1.3
        
        #calculate everything again
        country_sum = []
        for i in range(1,len(raw_matrix)):
            country_sum.append(np.sum(raw_matrix[i][1::]))
    
        product_sum = []
        for i in range(1,len(raw_matrix[0])):
            product = 0
            for j in range(1,len(raw_matrix)):
                product += raw_matrix[j][i]
            product_sum.append(product)
        total_sum = np.sum(product_sum)
    
        mcp = []
     
        for i in range (1,len(raw_matrix)):
            country_row = []
            for j in range(1,len(raw_matrix[i])):
                percentage = raw_matrix[i][j]/country_sum[i-1]
                total_percentage = product_sum[j-1]/total_sum
                if percentage > total_percentage:
                    country_row.append(1)
                else:
                    country_row.append(0)
            mcp.append(country_row)
    
        #compute mpp
        country_sum2 = []
        for i in range(0,len(mcp)):
            country_sum2.append(np.sum(mcp[i]))
    
        product_sum2 = []
        for i in range(0,len(mcp[0])):
            product = 0
            for j in range(0,len(mcp)):
                product += mcp[j][i]
            product_sum2.append(product)
    
        mpp = []
        for i in range(0,len(mcp[0])):
            product_row = []
            for j in range(0,len(mcp[0])):
                temp_sum = 0
                for k in range(0,len(mcp)):
                    try:
                        temp_sum += (mcp[k][i] * mcp[k][j]/(country_sum2[k]*product_sum2[i]))
                    except:
                        pass
                product_row.append(temp_sum)
            mpp.append(product_row)
    
        #compute PCI
        mpp = np.array(mpp)
    
        e_vals, e_vecs = LA.eig(mpp)
        eigenvectors = e_vecs.tolist()
        eigen = []
        for eigenvector in eigenvectors:
            temp = []
            for col in eigenvector:
                temp.append(col.real)
            eigen.append(temp)
    
        vector = []
        for i in range(0,len(eigen)):
            vector.append(eigen[i][1])
    
        pci = []
        if vector[len(vector)-2] > 0:
            for item in vector:
                pci.append((item-np.mean(vector))/np.std(vector))
        else:
            for item in vector:
                pci.append((-item+np.mean(vector))/np.std(vector))
    
        #compute proximity
        proximity = []
        for i in range(0,len(mcp[0])):
            product_row = []
            for j in range(0,len(mcp[0])):
                temp_sum = 0
                for k in range(0,len(mcp)):
                    try:
                        temp_sum += mcp[k][i] * mcp[k][j]
                    except:
                        pass
                p0 = product_sum2[i]
                p1 = product_sum2[j]
                if p0 < p1:
                    large = p1
                else:
                    large = p0
                product_row.append(temp_sum/large)
            proximity.append(product_row)
    
        #compute actual distance
        distance = []
        for i in range(0,len(mcp)):
            product_row = []
            for j in range(0,len(mcp[0])):
                sum1 = 0
                for k in range(0,len(mcp[0])):
                    try:
                        sum1 += (1-mcp[i][k])*proximity[j][k]
                    except:
                        pass
                sum2 = 0
                for k in range(0,len(mcp[0])):
                    try:
                        sum2 += proximity[j][k]
                    except:
                        pass
                if mcp[i][j] == 1:
                    product_row.append(0)
                else:
                    product_row.append(sum1/sum2)
            distance.append(product_row)
    
        #compute opportunity gain
        opportunity_gain = []
        for i in range(0,len(mcp)):
            country_row = []
            for j in range(0,len(mcp[0])):
                gain = 0
                for k in range(0,len(mcp[0])):
                    temp = 0
                    for l in range(0,len(mcp[0])):
                        temp += proximity[l][k]
                    gain += (proximity[j][k]/temp)*(1-mcp[i][k])*pci[k]
                gain = gain - (1-distance[i][j]) * pci[j]
                if mcp[i][j] == 1:
                    country_row.append(0)
                else:
                    country_row.append(gain)
            opportunity_gain.append(country_row)
        
        #compute marginal opportunity gain
        marginal = []
        for i in range(0,len(mcp)):
            country_row = []
            for j in range(0,len(mcp[0])):
                if mcp[i][j] == 1:
                    country_row.append(0)
                else:
                    export = raw_matrix[i+1][j+1]
                    country_export = country_sum[i]
                    product_export = product_sum[j]
                    export_increase = (product_export*country_export/total_sum)-export
                    country_row.append(opportunity_gain[i][j]*country_export/export_increase)
            marginal.append(country_row)
        
        #print("Iteration ", ite_round, ": ")
        #for i in range(0,len(marginal[country_index])):
        #    print(product_list[i], ": ", marginal[country_index][i])
        #print(" ")
        
        temp = []
        string = "Iteration " + str(ite_round)
        temp.append(string)
        posi = {}
        zero = []
        nega = {}
        color = [1]
        for i in range(0,len(marginal[country_index])):
            if marginal[country_index][i] > 0:
                posi[product_list[i]] = marginal[country_index][i]
            elif marginal[country_index][i] == 0:
                zero.append(product_list[i])
            else:
                nega[product_list[i]] = marginal[country_index][i]
        posi = rank(posi)
        nega = rank(nega)
        temp.extend(posi)
        color.extend([44 for i in range(len(posi))])
        temp.extend(zero)
        color.extend([27 for i in range(len(zero))])
        temp.extend(nega)
        color.extend([1 for i in range(len(nega))])
        strategy.append(temp)
        colors.append(color)
    
    strategy = np.array(strategy)
    strategy = np.transpose(strategy)
    colors = np.array(colors)
    colors = np.transpose(colors)
    #print(f'color shape is {colors.shape}')

    book = Workbook()
    sheet1 = book.add_sheet('Sheet 1')
    for i in range(0,len(strategy)):
        for j in range(len(strategy[0])):
          st = xlwt.easyxf('pattern: pattern solid;')
          st.pattern.pattern_fore_colour = colors[i][j]
          sheet1.write(i,j,strategy[i][j], st)
    book.save(f'sequential strategy of {country}.xls')
    
    book = Workbook()
    sheet1 = book.add_sheet('Sheet 1')
    for i in range(0,len(sdv)):
          st = xlwt.easyxf('pattern: pattern solid;')
          st.pattern.pattern_fore_colour = sdv_colors[i]
          sheet1.write(i, 0, sdv[i][0], st)
    book.save(f'sequential vector of {country}.xls')
    print(f'{country} completed')
    
def main():
    countries = ['Brazil', 'USA', 'Mexico', 'Argentina', 'Colombia',       
               'France', 'Germany', 'Spain', 'Greece', 'Portugal', 'Italy', 'Poland',
               'Turkey', 'Saudi Arabia', 'United Arab Emirates',
               'Morocco', 'South Africa', 'Egypt', 'Nigeria', 'Rwanda', 'Tunisia', 'Ethiopia', 'Algeria',
               'China', 'Republic of Korea', 'Japan', 'Malaysia', 'Thailand', 'Singapore', 'Indonesia']
               
    countries_ = ['Brazil']
    for country in countries_:
        sdv_and_sids(country, root_path+'2013-2018 cleaned/2015clean')

In [0]:
if __name__ == "__main__":
    main()   #Change the countries and year above to have different results

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:33: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison


Brazil completed


# SOI

In [0]:
def read_it(filename):
    f = open(filename,'r', encoding='cp1252')
    reader = csv.reader(f)
    raw_matrix = []
    for row in reader:
        temp = []
        for i in range(0,len(row)):
            try:
                temp.append(float(row[i]))
            except ValueError:
                #print(row[i])
                temp.append(row[i])
        raw_matrix.append(temp)  
    f.close()
    return raw_matrix

In [0]:
def compute_soi_adi(adi, country_list, product_list,country_list2, product_list2, opportunity_gain):
    soi = []
    if len(product_list) == len(product_list2):
        print("categorization verified")

    common = []
    for i in range(0,len(country_list)):
        if country_list[i] in country_list2:
            for j in range(0,len(country_list2)):
                if country_list[i] == country_list2[j]:
                    dependent = adi[country_list[i]]
                    independent = []
                    for item in opportunity_gain[i]:
                        independent.append([item])
                    reg = LinearRegression()
                    reg.fit(independent, dependent)
                    r_2 = reg.score(independent, dependent)
                    common.append(country_list[i])
                    # if country_list[i] == "China":
                    #      matrix = open(root_path+"soi analysis TEST ROHAN.csv",'w',newline='')
                    #      writer = csv.writer(matrix)
                    #      start = ["Product","OG","Export Increase"]
                    #      writer.writerow(start)
                    #      for i in range(0,len(independent)):
                    #          temp = [product_list[i],independent[i][0],dependent[i]]
                    #          writer.writerow(temp)
                    #      matrix.close()
                    soi.append(r_2)
                    
    matrix = open(root_path+"Rohan Testing/structual optimality index.csv",'w',newline='')
    writer = csv.writer(matrix)
    col_name = ["Country","SOI"]
    writer.writerow(col_name)
    for i in range(0,len(soi)):
        temp = [common[i],soi[i]]
        writer.writerow(temp)
    matrix.close()
    return soi


In [0]:
def compute_regress(gdp_raw, country_list, country_list2):
    #comprehensive regression
    common=np.intersect1d(country_list, country_list2)
    country_list3 = []
    gdp_change = []
    for i in range(1,len(gdp_raw)):
        country_list3.append(gdp_raw[i][0])
        gdp_change.append(float(gdp_raw[i][2])-float(gdp_raw[i][1]))

    independent = []
    dependent = []
    name = []

    for i in range(0,len(common)):
        if common[i] in country_list3:
            for j in range(0,len(country_list3)):
                if common[i] == country_list3[j]:
                    name.append(country_list3[j])
                    dependent.append(gdp_change[j])
                    independent.append(soi[i])

    data = np.array(independent)
    X = sm.add_constant(data)
    model = sm.OLS(dependent, X).fit()
    return model

In [0]:
soi= compute_soi_adi(adi, country_list, product_list,country_list2, product_list2, opportunity_gain)
gdp_raw= read_it(root_path+'GDP.csv')
reg= compute_regress(gdp_raw, country_list, country_list2)
reg.summary()

categorization verified


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.009
Model:                            OLS   Adj. R-squared:                 -0.004
Method:                 Least Squares   F-statistic:                    0.6744
Date:                Mon, 24 Feb 2020   Prob (F-statistic):              0.414
Time:                        03:47:00   Log-Likelihood:                -2116.7
No. Observations:                  74   AIC:                             4237.
Df Residuals:                      72   BIC:                             4242.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        2.31e+11   9.97e+10      2.317      0.023    3.23e+10     4.3e+11
x1         -4.801e+11   5.85e+11     -0.821      0.414   -1.65e+12    6.85e+11
==============================================================================
Omnibus:                      145.816   Durbin-Watson:                   1.928
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             9728.695
Skew:                           7.118   Prob(JB):                         0.00
Kurtosis:                      57.338   Cond. No.                         7.85
==============================================================================

Warnings:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [0]:
gdp_raw2=pd.read_csv(root_path+'GDP2.csv', index_col=0)

In [0]:
gdp_raw2

,Country Code,Indicator Name,1960,1961,1962,1963,1964,1965,1966,1967,1968,1969,1970,1971,1972,1973,1974,1975,1976,1977,1978,1979,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019
Country Name,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Aruba,ABW,"GDP, PPP (current international $)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.497860e+09,1.671820e+09,1.810503e+09,1.988839e+09,2.197955e+09,2.301201e+09,2.371124e+09,2.581983e+09,2.663056e+09,2.734950e+09,3.009056e+09,2.983690e+09,2.931664e+09,3.045100e+09,3.374482e+09,3.521849e+09,3.666548e+09,3.832821e+09,3.903830e+09,3.519787e+09,3.429585e+09,3.621880e+09,3.640776e+09,3.860199e+09,3.966592e+09,3.990945e+09,4.026065e+09,4.157177e+09,NaN,NaN
Afghanistan,AFG,"GDP, PPP (current international $)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.897303e+10,2.103225e+10,2.190391e+10,2.512257e+10,2.726947e+10,3.187365e+10,3.376901e+10,4.130488e+10,4.778771e+10,4.899395e+10,5.630125e+10,6.049792e+10,6.332195e+10,6.492790e+10,6.712153e+10,7.022035e+10,7.267225e+10,NaN
Angola,AGO,"GDP, PPP (current international $)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.660776e+10,3.822093e+10,3.680966e+10,2.864440e+10,2.964791e+10,3.481002e+10,4.024852e+10,4.392084e+10,4.649876e+10,4.819912e+10,5.078182e+10,5.407841e+10,6.244088e+10,6.550203e+10,7.463298e+10,8.852367e+10,1.017340e+11,1.191030e+11,1.349780e+11,1.371750e+11,1.455160e+11,1.537140e+11,1.700440e+11,1.816010e+11,1.939600e+11,1.978840e+11,1.948870e+11,1.982990e+11,1.987960e+11,NaN
Albania,ALB,"GDP, PPP (current international $)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.378950e+09,6.236675e+09,5.920351e+09,6.639981e+09,7.344851e+09,8.497883e+09,9.440948e+09,8.555020e+09,9.451335e+09,1.079260e+10,1.192777e+10,1.316288e+10,1.422189e+10,1.518142e+10,1.641442e+10,1.766329e+10,1.963046e+10,2.161074e+10,2.425160e+10,2.580553e+10,2.804664e+10,2.965551e+10,3.053030e+10,3.060405e+10,3.252907e+10,3.359485e+10,3.413408e+10,3.715420e+10,3.830669e+10,NaN
Andorra,AND,"GDP, PPP (current international $)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Kosovo,XKX,"GDP, PPP (current international $)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.269750e+09,8.135564e+09,8.206321e+09,8.858584e+09,9.334703e+09,1.020620e+10,1.098856e+10,1.210588e+10,1.266749e+10,1.322321e+10,1.381990e+10,1.472611e+10,1.543022e+10,1.624135e+10,1.675098e+10,1.762344e+10,1.854133e+10,1.969222e+10,2.094114e+10,NaN
"Yemen, Rep.",YEM,"GDP, PPP (current international $)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.603165e+10,2.860568e+10,3.165892e+10,3.370601e+10,3.673984e+10,3.963681e+10,4.223336e+10,4.520900e+10,4.846397e+10,5.101989e+10,5.538493e+10,5.875262e+10,6.203057e+10,6.555023e+10,6.998921e+10,7.620486e+10,8.100009e+10,8.595275e+10,9.082081e+10,9.505129e+10,1.035650e+11,9.228542e+10,9.630605e+10,1.027230e+11,1.044690e+11,8.797591e+10